# Project 08 — BROKEN notebook (debugging exercise)

Two seeded bugs: a **centered horseshoe** that floods the sampler with divergences, and a **double-dipping** analysis that fakes significance. Run it, read the divergence diagnostics, find each bug, and fix it. Clean reference: `notebook.ipynb`; answer key: `BROKEN_BUGS.md` (don't peek).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
X, y = data['X'], data['y']
n, p = X.shape

### Model — BUG 1: a CENTERED horseshoe. The scale funnel will cause divergences.

In [ ]:
# BUG 1: sampling beta DIRECTLY through tau*lambda (centered). The joint of
#        beta and its own scale is a pinched funnel that NUTS cannot explore;
#        expect many divergences and unreliable estimates.
with pm.Model() as model:
    beta0 = pm.Normal('beta0', 0.0, 5.0)
    sigma = pm.HalfNormal('sigma', 5.0)
    tau = pm.HalfCauchy('tau', beta=0.1)
    lam = pm.HalfCauchy('lam', beta=1.0, shape=p)
    beta = pm.Normal('beta', 0.0, tau * lam, shape=p)   # BUG 1: centered funnel
    mu = beta0 + pm.math.dot(X, beta)
    pm.Normal('y', mu=mu, sigma=sigma, observed=y)
    idata = pm.sample(draws=800, tune=800, chains=2, random_seed=RNG,
                      progressbar=False, target_accept=0.9,
                      idata_kwargs={'log_likelihood': True})

In [ ]:
# The divergence count is the alarm. Many divergences => the funnel.
print('divergences:', int(idata.sample_stats['diverging'].sum()))
print(az.summary(idata, var_names=['tau']))
az.plot_energy(idata); plt.tight_layout()  # mismatched marginal/energy => trouble

### Interpretation — BUG 2: double-dipping. Selecting the top coefficient and re-testing it on the SAME data fakes significance.

In [ ]:
# BUG 2: pick the largest-|coef| predictor, then refit a SIMPLE regression
#        on ONLY that predictor and brag about its narrow interval / 'p-value'.
#        This uses the data twice (to SELECT and to TEST) -> selection bias.
bmean = idata.posterior['beta'].mean(('chain','draw')).values
j_star = int(np.argmax(np.abs(bmean)))
with pm.Model() as cheat:
    a = pm.Normal('a', 0, 5); b = pm.Normal('b', 0, 5); s = pm.HalfNormal('s', 5)
    pm.Normal('y', mu=a + b * X[:, j_star], sigma=s, observed=y)
    idata2 = pm.sample(draws=500, tune=500, chains=2, random_seed=RNG,
                       progressbar=False)
bb = idata2.posterior['b'].values.ravel()
print(f'Selected predictor {j_star}; refit slope 94% interval:',
      np.round(np.percentile(bb, [3, 97]), 2),
      '<- this narrow interval is MISLEADING (double-dipped).')